In [ ]:
from google.colab import files
files.upload()

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC

In [ ]:
df = pd.read_csv('p8_moons.csv')
X = df[['x1', 'x2']].values
y = df.target.values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

def show(m, ax, t):
    m.fit(Xtr, ytr)
    gx, gy = np.meshgrid(np.linspace(-2.5, 3.5, 200), np.linspace(-1.5, 2, 200))
    Z = m.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, Z, levels=[-100, 0, 100], alpha=.2)
    ax.contour(gx, gy, Z, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'])
    ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, edgecolor='k', s=20)
    ax.scatter(*m.support_vectors_.T, s=120, facecolors='none', edgecolors='r')
    ax.set_title('%s acc=%.3f SV=%d' % (t, m.score(Xte, yte), len(m.support_vectors_)))

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
show(SVC(kernel='linear', C=1), ax[0], 'Linear')
show(SVC(kernel='rbf', C=1, gamma=1), ax[1], 'RBF')
show(SVC(kernel='poly', degree=3, C=1), ax[2], 'Poly')
plt.show()

In [ ]:
Cs = [0.1, 1, 10, 100]
gs = [0.01, 0.1, 1, 10]
g = GridSearchCV(SVC(kernel='rbf'), {'C': Cs, 'gamma': gs}, cv=3).fit(Xtr, ytr)
print(g.best_params_, 'cv=%.3f test=%.3f' % (g.best_score_, g.score(Xte, yte)))
S = g.cv_results_['mean_test_score'].reshape(len(Cs), len(gs))
plt.imshow(S, cmap='viridis')
plt.xticks(range(len(gs)), gs)
plt.yticks(range(len(Cs)), Cs)
plt.xlabel('gamma')
plt.ylabel('C')
plt.colorbar()
for i in range(len(Cs)):
    for j in range(len(gs)):
        plt.text(j, i, '%.2f' % S[i, j], ha='center', color='w')
plt.show()